In [1]:
import pandas as pd
from Bio import SeqIO
import numpy as np

In [6]:
# Load allele table
alleles_no_dif = pd.read_csv(
    "../data/allele_sequences.no_differentiation.csv"
)

# Keep only HLA-A / HLA-B / HLA-C
alleles = alleles_no_dif[
    alleles_no_dif["normalized_allele"].str.startswith(("HLA-A", "HLA-B", "HLA-C"))
].reset_index(drop=True)


prefered_allels = np.concatenate((pd.read_csv('../data/train_pairs.csv').allele.unique(),
                                  pd.read_csv('../data/val_pairs.csv').allele.unique()))
def pick_row(group):
    pref_rows = group[group["normalized_allele"].isin(prefered_allels)]
    if len(pref_rows) > 0:
        return pref_rows.iloc[0]
    else:
        return group.iloc[0]

alleles = (
    alleles
    .groupby("sequence", sort=False, group_keys=False)
    .apply(pick_row)
    .reset_index(drop=True)
)

# Remove duplicate pseudo-sequences
alleles = alleles[~alleles["sequence"].duplicated()].reset_index(drop=True)

In [7]:
# Load full and aligned class I sequences from FASTA
full_seq_dict = {}
for record in SeqIO.parse("../data/class1.fasta", "fasta"):
    allele_name = record.description.split()[1]
    full_seq_dict[allele_name] = str(record.seq)

full_seq_aligned_dict = {}
for record in SeqIO.parse("../data/class1.aligned.fasta", "fasta"):
    allele_name = record.description.split()[1]
    full_seq_aligned_dict[allele_name] = str(record.seq)

In [8]:
# Add full sequence columns
alleles["full_seq"] = alleles["normalized_allele"].map(full_seq_dict)
alleles["full_seq_aligned"] = alleles["normalized_allele"].map(full_seq_aligned_dict)

# Keep only rows with both sequences available
alleles = alleles.dropna(subset=["full_seq", "full_seq_aligned"]).reset_index(drop=True)

In [9]:
# Raw residue positions of interest
raw_positions = [
    30, 32, 47, 68, 82, 85, 86, 89, 90, 92, 93, 96, 97, 99, 100, 103, 104,
    107, 118, 120, 122, 137, 139, 141, 166, 170, 173, 175, 179, 181, 182,
    186, 190, 194
]

In [10]:
from Bio.Seq import Seq
def raw_positions_to_aligned_positions(full_seq, aligned_seq, raw_positions):
    aligned_seq = Seq(aligned_seq)
    ungapped_reference = "".join(residue for residue in aligned_seq if residue != "-")

    if ungapped_reference != full_seq:
        raise ValueError("Aligned reference does not match the raw reference sequence.")

    non_gap_columns = [i for i, residue in enumerate(aligned_seq) if residue != "-"]
    return [non_gap_columns[pos] for pos in raw_positions]
idx = alleles.index[alleles.normalized_allele == "HLA-A*01:01"][0]
aligned_positions = raw_positions_to_aligned_positions(
    alleles.loc[0, "full_seq"],
    alleles.loc[0, "full_seq_aligned"],
    raw_positions,
)


In [11]:
# For one aligned sequence, recover the corresponding raw indices
def get_raw_positions_from_aligned(aligned_seq, aligned_positions):
    raw_indices = []
    raw_index = 0
    target_idx = 0

    for i, char in enumerate(aligned_seq):
        if i == aligned_positions[target_idx]:
            raw_indices.append(raw_index)
            target_idx += 1

            if target_idx == len(aligned_positions):
                break

        if char != "-":
            raw_index += 1

    return raw_indices
# Reconstruct pseudo-sequence from full sequence
# Store the raw positions for each allele
alleles["poses"] = alleles["full_seq_aligned"].apply(
    lambda seq: get_raw_positions_from_aligned(seq, aligned_positions)
)
reconstructed_sequence = alleles.apply(
    lambda row: "".join(row["full_seq"][i] for i in row["poses"]),
    axis=1
)

# Identify problematic alleles
bad_mask = alleles["sequence"] != reconstructed_sequence

deleted_alleles = alleles.loc[bad_mask, "normalized_allele"].tolist()

print("Deleted alleles:", deleted_alleles)
print("Number deleted:", len(deleted_alleles))

# Keep only valid ones
alleles = alleles.loc[~bad_mask].reset_index(drop=True)

print("Remaining alleles:", len(alleles))

Deleted alleles: []
Number deleted: 0
Remaining alleles: 4608


In [ ]:
from transformers import T5Tokenizer, T5EncoderModel
import os
import torch
import re
from tqdm import tqdm
#os.environ['HF_HOME']=...
#os.environ["TRANSFORMERS_CACHE"] = ...

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Load the tokenizer
tokenizer = T5Tokenizer.from_pretrained('Rostlab/ProstT5',do_lower_case=False)

# Load the model
model = T5EncoderModel.from_pretrained("Rostlab/ProstT5").to(device)
model.full() if device=='cpu' else model.half()



In [ ]:
torch.cuda.empty_cache()
alleles_seq=alleles.full_seq.values
positions=alleles.poses.values
sequence_examples = [" ".join(list(re.sub(r"[UZOB]", "X", sequence))) for sequence in alleles_seq]
sequence_examples = [ "<AA2fold>" + " " + s if s.isupper() else "<fold2AA>" + " " + s
                      for s in sequence_examples
                    ]


In [ ]:
import math
batch_size = 50
batch_num = math.ceil(len(sequence_examples)/batch_size)
p = torch.zeros((0,34,1024),dtype=torch.float16)

for i in tqdm(range(batch_num)):
    batch=sequence_examples[i*batch_size:(i+1)*batch_size]
    poses=positions[i*batch_size:(i+1)*batch_size]
    ids = tokenizer.batch_encode_plus(batch, add_special_tokens=True, padding="longest",return_tensors='pt').to(device)
    with torch.no_grad():
        embedding_rpr = model(
                  ids.input_ids, 
                  attention_mask=ids.attention_mask
                  )
    for index in range(embedding_rpr.last_hidden_state.shape[0]):
        embedding = embedding_rpr.last_hidden_state[index:index+1,1:-1,:].to('cpu')
        embedding = embedding[:,poses[index],:]
        p=torch.concat((p,embedding),dim=0)

In [ ]:
embdeding_dict=dict()
psudo_seq=alleles.sequence.values
for peptide,embs in zip(psudo_seq,p):
    embdeding_dict[peptide]=embs

In [ ]:
import pickle
with open('peptide_embdeding.pkl', 'wb') as file:
    pickle.dump(embdeding_dict, file)

In [ ]:
alleles_no_dif=alleles_no_dif[alleles_no_dif.sequence.isin(alleles.sequence)].reset_index(drop=True)
d = dict(zip(alleles_no_dif.iloc[:, 0], alleles_no_dif.iloc[:, 1]))
with open('allele_mapping.pkl', 'wb') as file:
    pickle.dump(d, file)